# Введение

Подключение диска

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install moviepy pydub

Загрузка видеозаписи и извлечение из неё кадров и аудио

In [ ]:
from moviepy.editor import VideoFileClip
from pydub import AudioSegment
import os

def extract_audio(video_path, output_audio_path):
    # Извлечение аудио из видео
    clip = VideoFileClip(video_path)
    clip.audio.write_audiofile(output_audio_path, codec='pcm_s16le')
    clip.close()

def extract_frames(video_path, output_frames_dir, interval=1):
    # Извлечение кадров из видео с интервалом в одну секунду
    clip = VideoFileClip(video_path)
    duration = int(clip.duration)
    if not os.path.exists(output_frames_dir):
        os.makedirs(output_frames_dir)
    for i in range(0, duration, interval):
        frame = clip.get_frame(i)
        frame_image_path = os.path.join(output_frames_dir, f"frame_{i}.jpg")
        clip.save_frame(frame_image_path, t=i)
    clip.close()

# Задаём путь к видео
video_path = '/content/drive/MyDrive/original (№1).mp4'
output_audio_path = '/content/output/audio.wav'
output_frames_dir = '/content/output/frames/'

# Извлекаем аудио и кадры
extract_frames(video_path, output_frames_dir)
extract_audio(video_path, output_audio_path)



# Видеомодель

Подключение необходимых библиотек

In [ ]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Класс для работы с датасетом

In [ ]:
from PIL import Image
from typing import Tuple

class ImageDataset:
    def __init__(self, root_dir, transform=None):
        """
        Инициализация класса.

        :param root_dir: Путь к папке с подкаталогами, где каждый подкаталог представляет класс и содержит изображения.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []

        # Сканируем папку и создаём список изображений и их классов

        for image_name in os.listdir(root_dir):
            image_path = os.path.join(root_dir, image_name)
            self.image_paths.append(image_path)

    def __len__(self) -> int:
        """Возвращает количество изображений в датасете."""
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, int]:
        """
        Возвращает изображение и номер его класса по индексу.

        :param idx: Индекс изображения.
        :return: Кортеж (изображение, номер класса).
        """
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

dir = '/content/output/frames'

Архитектура видеомодели

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        batch, channels, _, _ = x.size()
        y = self.global_pool(x).view(batch, channels)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(batch, channels, 1, 1)
        return x * y


class SkipConnectionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SkipConnectionBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.dropout1 = nn.Dropout2d(0.3)  # Добавлено dropout

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout2 = nn.Dropout2d(0.3)  # Добавлено dropout

        self.se = SEBlock(out_channels)  # Добавляем SEBlock

        # Используем skip_conv только если количество каналов не совпадает
        self.use_skip_conv = in_channels != out_channels
        if self.use_skip_conv:
            self.skip_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        identity = self.skip_conv(x) if self.use_skip_conv else x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout1(out)  # Dropout после первого слоя
        out = self.bn2(self.conv2(out))
        out = self.dropout2(out)  # Dropout после второго слоя
        out = self.se(out)  # Применяем SEBlock
        out += identity
        return F.relu(out)


class MyModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            SkipConnectionBlock(3, 16),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(16, 64),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(64, 128),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(128, 256),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(256, 512),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(512, 1024),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



model = MyModel(num_classes=6).to(device)
summary(model, (3, 180, 180))

Загрузка кадров как датасета

In [ ]:
transform = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

images_ = ImageDataset(root_dir=dir, transform=transform)
infer_dataloader = DataLoader(images_, batch_size=64, shuffle=False)

In [ ]:
best_model_path = '/content/drive/MyDrive/best_model1.pth'

# Инициализация модели и загрузка в неё лучшие после обучения веса

model = MyModel(num_classes=6).to(device)
model.load_state_dict(torch.load(best_model_path))

model.eval()

In [ ]:
len(images_)

Инференс видеомодели и получение числовых характеристик предсказаний, а также их нормализация

In [ ]:
def min_max_normalize(tensor):
    min_val = tensor.min()
    max_val = tensor.max()
    normalized_tensor = (tensor - min_val) / (max_val - min_val)
    return normalized_tensor

results_img = []
sft_img = []
for images in tqdm(infer_dataloader):
    images = images.to(device)

    with torch.no_grad():
        outputs = model(images)
        #preds = torch.argmax(outputs).cpu().numpy()
        preds = outputs

    for pred_class in preds:
        results_img.append(min_max_normalize(pred_class))
        sft_img.append(torch.softmax(pred_class, dim=0))

results_img

In [ ]:
def num2acc(lst):
    accs = ['Am', 'C', 'Dm', 'Em', 'F', 'G']
    for i in range(len(lst)):
        lst[i] = accs[lst[i]]
    return lst

res_img = []
for i in range(len(results_img)):
    res_img.append(torch.argmax(results_img[i]).cpu().numpy())

res_img = [item.item() for item in res_img]
res_img = num2acc(res_img)
print(res_img)

# Аудио

Подключение необходимых библиотек

In [ ]:
import random
from pathlib import Path
import os
import librosa

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F

from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from glob import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Константы для работы с аудио

In [ ]:
path = '/content/output/'
name = 'audio.wav'
sr = 16000
dt = 1.0
N_CLASSES = 6
n_mels = 64
hop_length = 256
n_fft = 1024
threshold=0.01
frames = (sr//hop_length)+1

Обработка аудио

In [ ]:
def envelope(y):
    mask = []
    y1 = pd.Series(y).apply(np.abs)
    y_mean = y1.rolling(window=int(sr/20),
                       min_periods=1,
                       center=True).max()
    for mean in y_mean:
        if mean > threshold:
            mask.append(True)
        else:
            mask.append(False)
    return mask

In [ ]:
def load_audio(path, sr):
    rate = librosa.get_samplerate(path)
    y1, _ = librosa.load(path, sr=rate)
    wav = librosa.resample(y=y1, orig_sr=rate, target_sr=sr)
    return wav

In [ ]:
def get_melspectogram(waveform):
    mel_spectrogram = librosa.feature.melspectrogram(y=waveform, sr=sr, n_fft=n_fft,
                                                     hop_length=hop_length, n_mels=n_mels)
    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
    return log_mel_spectrogram

In [ ]:
class AudDataset:
    def __init__(self, root_dir, wav_name=None, n_classes=6):
        self.root_dir = root_dir
        self.sr = sr
        self.dt = dt
        self.n_mels = n_mels
        self.frames = frames

        self.wav_paths = []

        if (wav_name):
            wav_path = os.path.join(root_dir, wav_name)
            self.wav_paths.append(wav_path)
        else:
            for wavname in os.listdir(root_dir):
                wav_path = os.path.join(root_dir, wavname)
                self.wav_paths.append(wav_path)


        self.X = self._prepare_data()

    def _prepare_data(self):
        X = np.empty((1, int(self.sr * self.dt)), dtype=np.float32)

        for file in self.wav_paths:
            y1 = load_audio(file, self.sr)
            mask = envelope(y1)
            y = y1[mask]
            n = y.shape[0]
            m = n // self.sr
            for k in range(m):
                a = np.expand_dims(y[k * self.sr:(k + 1) * self.sr], axis=0)
                X = np.append(X, a, axis=0)

        X = np.delete(X, 0, axis=0)

        n_samples = X.shape[0]
        X_mel = np.empty((n_samples, self.n_mels, self.frames), dtype=np.float32)

        for i in np.arange(n_samples):
            X_mel[i] = get_melspectogram(X[i])

        return X_mel

    def __getitem__(self, idx):
        return self.X[idx]

    def __len__(self):
        return len(self.X)

Архитектура аудиомодели

In [ ]:
class Conv2DModel(nn.Module):
    def __init__(self, n_classes):
        super(Conv2DModel, self).__init__()

        # Нормализация слоев
        self.layer_norm = nn.LayerNorm([n_mels, frames])

        # Сверточные слои
        self.conv1 = nn.Conv2d(1, 8, kernel_size=(7, 7), padding='same')
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=(5, 5), padding='same')
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv3 = nn.Conv2d(16, 16, kernel_size=(3, 3), padding='same')
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=(3, 3), padding='same')
        self.pool4 = nn.MaxPool2d(kernel_size=(2, 2), padding=0)

        self.conv5 = nn.Conv2d(32, 32, kernel_size=(3, 3), padding='same')

        # Полносвязные слои
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(32 * (n_mels // 16) * (frames // 16), 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # Применение LayerNorm к формату последнего канала и восстановление формата первого канала
        x = self.layer_norm(x.squeeze(1)).unsqueeze(1)

        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        x = F.relu(self.conv3(x))
        x = self.pool3(x)

        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        x = F.relu(self.conv5(x))

        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return F.softmax(x, dim=1)


model = Conv2DModel(n_classes=6).to(device)

summary(model, (64, 63))

Загрузка аудиозаписи как датасета

In [ ]:
audio = AudDataset(root_dir=path, wav_name=name)
audio = torch.tensor(audio)
audio_dataloader = DataLoader(audio, batch_size=32, shuffle=False)

In [ ]:
best_model_path = '/content/drive/MyDrive/best_model_aud.pth'

# Инициализация модели и загрузка в неё лучшие после обучения веса

model = Conv2DModel(n_classes=6).to(device)

model.load_state_dict(torch.load(best_model_path))
model.eval()

In [ ]:
len(audio)

Инференс аудмомодели и получение числовых характеристик предсказаний, а также их нормализация

In [ ]:
def min_max_normalize(tensor):
    min_val = tensor.min()
    max_val = tensor.max()
    normalized_tensor = (tensor - min_val) / (max_val - min_val)
    return normalized_tensor

results_aud = []
sft_aud = []

for wav in tqdm(audio_dataloader):
    wav = wav.to(device)

    with torch.no_grad():
        outputs = model((wav))
        #preds = torch.argmax(outputs, dim=1).cpu().numpy()
        preds = outputs

    for pred_class in preds:
        results_aud.append(min_max_normalize(pred_class))
        sft_aud.append(pred_class)

results_aud

In [ ]:
if (len(results_aud) < len(results_img)):
    results_img = results_img[:len(results_aud)-len(results_img)]

# Мета-модель

In [ ]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
class MetaModel(nn.Module):
    def __init__(self, input_size=12, num_classes=6):
        super(MetaModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)  # Скрытый слой
        self.relu = nn.ReLU()  # Активация
        self.fc2 = nn.Linear(64, num_classes)  # Выходной слой

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MetaModel().to(device)
summary(model, (12, ))

In [ ]:
best_model_path = '/content/drive/MyDrive/best_metamodel.pth'

model = MetaModel(num_classes=6).to(device)
model.load_state_dict(torch.load(best_model_path))

model.eval()

In [ ]:
def meta_combine(prob1, prob2):

    prob1 = torch.softmax(prob1, dim=0)
    prob2 = torch.softmax(prob2, dim=0)

    input = torch.tensor(torch.cat([prob1, prob2]), dtype=torch.float32).unsqueeze(0).to(device)
    output = model(input)
    probabilities = torch.softmax(output, dim=1)

    return torch.Tensor(probabilities)

# Результаты

Интеграция (суммирование) результатов моделей и получение предсказаний на счёт аккордов

In [ ]:
def dif_combine(prob1, prob2, epsilon=1e-8):
    def get_confidence(prob):
        sorted_prob, _ = torch.sort(prob, descending=True)
        return sorted_prob[0] - sorted_prob[1]

    conf1 = get_confidence(prob1)
    conf2 = get_confidence(prob2)

    total_conf = conf1 + conf2 + epsilon
    weight1 = conf1 / total_conf
    weight2 = conf2 / total_conf

    combined_prob = [weight1 * p1 + weight2 * p2 for p1, p2 in zip(prob1, prob2)]
    return torch.Tensor(combined_prob)


def ent_combine(prob1, prob2, epsilon=1e-8):
    def entropy(prob):
        return -sum(p * np.log(p + epsilon) for p in prob.cpu())

    ent1 = entropy(prob1)
    ent2 = entropy(prob2)

    conf1 = 1 / (ent1 + epsilon)
    conf2 = 1 / (ent2 + epsilon)

    total_conf = conf1 + conf2
    weight1 = conf1 / total_conf
    weight2 = conf2 / total_conf

    combined_prob = [weight1 * p1 + weight2 * p2 for p1, p2 in zip(prob1, prob2)]
    return torch.Tensor(combined_prob)

In [ ]:
def num2acc(lst):
    accs = ['Am', 'C', 'Dm', 'Em', 'F', 'G']
    for i in range(len(lst)):
        lst[i] = accs[lst[i]]
    return lst

results_sum = [tensor1 + tensor2 for tensor1, tensor2 in zip(results_img, results_aud)]


res_img = []
res_aud = []
res_sum = []
res_dif = []
res_ent = []
res_met = []

for i in range(len(results_img)):
    res_img.append(torch.argmax(results_img[i]).cpu().numpy())
    res_aud.append(torch.argmax(results_aud[i]).cpu().numpy())
    res_sum.append(torch.argmax(results_sum[i]).cpu().numpy())
    res_dif.append(torch.argmax(dif_combine(results_img[i], results_aud[i])).cpu().numpy())
    res_ent.append(torch.argmax(ent_combine(results_img[i], results_aud[i])).cpu().numpy())
    res_met.append(torch.argmax(meta_combine(sft_img[i], sft_aud[i])).cpu().numpy())


res_img = [item.item() for item in res_img]
res_aud = [item.item() for item in res_aud]
res_sum = [item.item() for item in res_sum]
res_dif = [item.item() for item in res_dif]
res_ent = [item.item() for item in res_ent]
res_met = [item.item() for item in res_met]


res_img = num2acc(res_img)
res_aud = num2acc(res_aud)
res_sum = num2acc(res_sum)
res_dif = num2acc(res_dif)
res_ent = num2acc(res_ent)
res_met = num2acc(res_met)

print(res_img)
print(res_aud)
print(res_sum)
print(res_dif)
print(res_ent)
print(res_met)

In [ ]:
results_img[1], results_aud[1]

Настройка среды для продвинутой работы с видео

In [ ]:
!apt-get install -y imagemagick
!pip install moviepy

In [ ]:
from moviepy.config import change_settings

change_settings({"IMAGEMAGICK_BINARY": "/usr/bin/convert"})


In [ ]:
!sed -i 's/rights="none"/rights="read|write"/' /etc/ImageMagick-6/policy.xml


Получение из исходной видеозаписи видео с легендой, демонстрирующей предсказания моделей и интеграции

In [ ]:
from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip
from IPython.display import display, Video

def play_video_with_legend(video_path, legends, output_path, lenth):
    # Загрузка видео
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    clip = VideoFileClip(video_path)

    # Создание текстовых клипов для каждой легенды
    text_clips = []
    duration = duration = clip.duration / lenth


    for i, (legend_img, legend_aud, legend_sum, legend_dif, legend_ent, legend_met) in enumerate(legends):
        legend = 'Video-model: ' + str(legend_img) + '\nAudiomodel: ' + str(legend_aud) + '\nSummation: ' + str(legend_sum) + '\nDifference: ' + str(legend_dif) + '\nEntropy: ' + str(legend_ent)+ '\nMeta-model: ' + str(legend_met)
        text = TextClip(legend, fontsize=42, color='black', font='Arial', align='NorthWest', size=clip.size, method='caption', stroke_color='black', stroke_width=2)
        text = text.set_position(('left', 'top')).set_duration(duration).set_start(i * duration)

        text_clips.append(text)

    # Объединение текста с оригинальным видео
    final_clip = CompositeVideoClip([clip] + text_clips)

    # Сохранение финального видео с легендой с параметрами для ускорения записи
    final_clip.write_videofile(output_path, codec='libx264', audio_codec='aac', preset='ultrafast', threads=8, fps=clip.fps)

    # Воспроизведение финального видео в Colab
    display(Video(output_path, width=320, height=320, embed=True))

legends = zip(res_img, res_aud, res_sum, res_dif, res_ent, res_met)
output_path = '/content/drive/MyDrive/datasets/video/output/video_with_legend.mp4'

# Воспроизведение видео с легендой
play_video_with_legend(video_path, legends, output_path, len(results_img))